<div style="background-color: #0596A6; padding: 20px; width: 100%; text-align: center;">

  <div style="max-width: 100%; width: 100%; margin: auto;">
    <svg width="100%" height="100" xmlns="http://www.w3.org/2000/svg">
      <rect width="100%" height="100" fill="#0596A6"/>
      <text x="50%" y="50%" dominant-baseline="middle" text-anchor="middle" font-size="40" fill="#FFFFFF" font-family="Arial, sans-serif">
        Big Data Management for Data Science
      </text>
    </svg>
  </div>

  <h1 style="color: #FFFFFF;"><b>Lab 1 - Graph Databases</b></h1>

  <h3 style="color: #FFFFFF;"> Team 1: Aleksandr Smolin · Julian Romero </h3>

  <h4 style="color: #FFFFFF;">April, 2025</h4>

  <hr style="width: max-width; margin: 20px auto;">

</div>


In [60]:
import polars as pl
import random
import numpy as np
from tqdm import tqdm
from neo4j import GraphDatabase
from datetime import datetime, timedelta

pl.Config.set_tbl_cols(-1) 

polars.config.Config

In [61]:
# General settings
DATA = '../data/files'
MISSING_THRESHOLD = 0.5
NUMBER_OF_JOURNALS = 250*1000
NUMBER_OF_CONFERENCES = 750*1000
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"

# A.1.
---

This section is in the Lab Document.

![BDM_lab01_a1.png](imgs/BDM_lab01_a1.png)

# A.2.
---

In this section, we do the following:
1. Understand the data generated from `dblp.xml` in section **Raw data loading, inspection and cleaning**. 
    - For each file, we inspect the columns and keep only relevant ones, ensure integrity of data (e.g. we only keep papers with existing authors) and handle missing data.
    - `dblp_article.csv`: this file contains scientific papers published in journals.
    - `dblp_author.csv`: contains author names and ids. We only keep those that have a published paper.
    - `dblp_inproceedings.csv`: contains those papers published in conferences.
    - `dblp_journal.csv`: contains journals and their ids.
    - `dblp_journal_published_in.csv`: Relationship: which article was published in which journal.
    - `dblp_proceedings.csv`:contains ConferenceEditions. Here, we distinguish those that are workshops from conferences.
2. Construct the final datasets based on the csv mentioned in 1 and on the schema presented in `A.1`:
    - Nodes:
        - `AUTHOR`: to create this, we concatenated all papers (i.e. those in conferences, workshops and journals). Then, we explode the list of authors, ending up with one author per row. Following, we keep those unique and assign the ids from `dblp_author.csv`. Finally, we generate fake data for country_of_birth, date_of_birth and is_phd.
        - `PAPER`: we concatenate `dblp_article.csv` and `dblp_inproceedings.csv` with the common columns, to have all papers, regadless where they were presented. Finally, we keep those unique, and create a fake alias.
        - `KEYWORD`: we generated a list of fake keywords and assigned and id to each of them.
        - `JOURNAL`: based on `dblp_journal.csv` we join it with `dblp_article.csv` to ensure we only keep those journals that have papers in the database.
        - `JORUNAL_VOLUME`: starting from `dblp_article.csv` (i.e. those papers published in journals), we kept all jorunal names, volume and dates, and created an id.
        - `WORKSHOP`: this is just a set of generated ids for those papers that where presented in conferences of the type workshop identified at the beggining.
        - `CONFERENCE`: this is just a set of generated ids for those papers that where presented in conferences of the type conference identified at the beggining.
        - `CONFERENCE_EDITION`: starting from `dblp_proceedings.csv`, we keep the different names and ids.
        - `VENUE`: to construct this dataset, we fakely assigned cities and their corresponding country to the conferences editions. Then, we only kept the date, city and country and created an id to be able to be maped from the conference edition.
        - `REVIEW`: to create this, we first keep the 50 most famous authors (in this case, considered as those with more publications). Then, we assigned to each paper a random set of these famous authors (between 1 to 5) as their reviewers exluding the authors of the papers themselves. Finally, we created a set of review id for each reviewer for each paper and keep only these ids. Then, in section `A.3` we will expand the properties of these nodes.
    - Edges:
        - `AUTHORED`: this edge connects Author ->[AUTHORED]-> Paper.
        - `CORRESPONDS_TO`: this edge connects Author <- [CORRESPONDS_TO] <- Paper. As in the description of the assignment it states that only one of the authors correspond the paper, we deliberatedly assigned one of the authors as the correspondant.
        - `CITES`: this edge connects Paper -> [CITES] -> Paper. We generated random citations for only 0.5% of the articles, given the computational complexity. For this, we generated the citations following a power law distribution (as a graph of citations will do in real life), ensuring that no paper can cite itself.
        - `HAS_KEYWORD`: this edge connects Paper -> [HAS_KEYWORD] -> Keyword. For the fake keywords created, we randomly assigned between 1 and 3 to each paper, ending up with a dataframe with paper ids and keywords ids.
        - `HAS_VOLUME`: this edge connects Journal -> [HAS_VOLUME] -> JournalVolume. Starting from `dblp_article.csv` (i.e. those papers published in journals), we kept all jorunal names and dates, created an id for this and join with the `df_node_jorunal`. Finally, we only kept the ids for the journal and the journal_volume.
        - `PUBLISHED_IN_J`: this edge connects Paper ->[PUBLISHED_IN_J]-> JorunalVolume. Starting from `dblp_article.csv`, we keep the id, journal, year, name and volume. Then, we filter only those papers that appear in `JournalVolume`. Finally, we keep the paper id and the jorunal volume id.
        - `PUBLISHED_IN_C`: this edge connects Paper ->[PUBLISHED_IN_C]-> ConferenceEdition. Starting from `dblp_proceedings.csv`, we filter only those papers that appear in `ConferenceEdition`. Finally, we keep the paper id and the conference edition id.
        - `HAS_EDITION_W`: this edge connects Workshop ->[HAS_EDITION_W]-> ConferenceEdition. We create a dataframe that contains the ids of the papers presented in workshop and the conference edition.
        - `HAS_EDITION_C`: this edge connects Conference ->[HAS_EDITION_C]-> ConferenceEdition. We create a dataframe that contains the ids of the papers presented in conferences and the conference edition.
        - `HELD_AT`: this edge connects ConferenceEdition ->[HELD_AT]-> Venue. This dataset contains the ids of the conference editions and maps to the venue where they were held at.
        - `WRITES`: this edge connects Author ->[WRITES]-> Review. 
        - `OF`: this edge connects Review ->[OF]-> Paper.


## General Functions Functions

In [3]:
def drop_many_nulls(df: pl.DataFrame, threshold: float = 0.5) -> pl.DataFrame:
    return df.select(
        [col for col in df.columns if (df.null_count().select(pl.col(col)).item() / df.height) <= threshold]
    )

def schema_type(columns: list[str]) -> dict[str, pl.DataType]:
    data_types = {
        'ID': pl.Int32,
        'int': pl.Int32,
        'string': pl.Utf8,
        'string[]': pl.List(pl.Utf8),
        'float': pl.Float32,
        'str': pl.Utf8,
        'boolean': pl.Boolean,
        'date': pl.Date,
    }
    return {col: data_types[col.split(':')[-1]] for col in columns}

countries   = ["United States","United Kingdom","Canada","Australia",
               "Germany","France","Spain","Italy","Netherlands","Sweden"]

def random_date(start_year=1950, end_year=1995):
    start = datetime(start_year, 1, 1)
    end   = datetime(end_year, 12, 31)
    delta = end - start
    return (start + timedelta(days=random.randrange(delta.days))).date()

def power_law_distribution(n, alpha=2.5):
    return [int(random.paretovariate(alpha)) for _ in range(n)]

def generate_citations(df: pl.DataFrame) -> pl.DataFrame:
    n = df.shape[0]  # Number of papers
    citing_ids = df.sample(n=int(n * 0.005), seed=42)[":ID"].to_list()  # Only 5% of papers will be citing
    print("Citing ids:", len(citing_ids))
    
    citations_per_paper = power_law_distribution(len(citing_ids), alpha=2.5)
    all_ids = np.array(df[":ID"].to_list())  # All paper IDs

    citation_pairs = []

    # Step 1: Loop over each citing paper to generate citations
    for citing_paper_id, num_citations in tqdm(zip(citing_ids, citations_per_paper), total=len(citing_ids), desc="Generating Citations"):
        num_citations = max(1, num_citations)  # Ensure at least 1 citation
        
        # Step 2: Create the valid citation IDs (excluding the current citing paper)
        valid_citations_ids = all_ids[all_ids != citing_paper_id]

        # Step 3: Randomly sample from the valid citation IDs
        sampled_indices = np.random.choice(valid_citations_ids, size=num_citations, replace=False)

        # Step 4: Append citation pairs
        for cited_paper_id in sampled_indices:
            citation_pairs.append((citing_paper_id, cited_paper_id))

    # Step 5: Create the final DataFrame with citation pairs
    citation_df = pl.DataFrame(citation_pairs, schema=[":START_ID", ":END_ID"], orient="row")
    return citation_df


## Raw data loading, inspection and cleaning

In [4]:
# ===================== AUTHORS =============================
df_aut = pl.read_csv(
        f'{DATA}/dblp_author.csv',
        ignore_errors=True,
        separator=';',
    )
df_aut = (df_aut
        .pipe(drop_many_nulls, threshold=MISSING_THRESHOLD)
        .unique(subset=[":ID"])
        .rename({"author:string": "name:string"})
    )
df_aut

:ID,name:string
i64,str
12453400,"""Jiangfeng Wu"""
14277902,"""Daniel Mailman"""
14338317,"""Pasquale Cennamo"""
14903269,"""Leonid Rebezyuk"""
12140124,"""Bo Tang 0015"""
…,…
11957473,"""Feridun Kaya"""
12680260,"""Tathagata Bandyopadhyay"""
13411809,"""Brien Flewelling"""


In [5]:
# ===================== Conference Editions =============================
col_names = pl.read_csv(f'{DATA}/dblp_proceedings_header.csv',separator=';').columns

df_conf_edition = pl.read_csv(
        f'{DATA}/dblp_proceedings.csv',
        ignore_errors=True,
        separator=';',
        new_columns=col_names,
    )

df_conf_edition = (
    df_conf_edition
    .pipe(drop_many_nulls, threshold=MISSING_THRESHOLD)
    .rename({"editor:string[]": "author:string[]",
             "proceedings:ID": ":ID",})
    .with_columns(
                pl.when(pl.col("title:string").str.to_lowercase().str.contains("workshop"))
                .then(pl.lit("workshop"))
                .otherwise(pl.lit("conference"))
                .alias("conf_type:string"),
                pl.col("author:string[]").str.split("|")
                ).explode("author:string[]")
    .join(
        df_aut.rename({":ID": "author_id:ID"}),
        left_on="author:string[]",
        right_on="name:string",
        how="left",
    )

)  

df_conf_edition.head()

:ID,booktitle:string,author:string[],ee:string[],isbn:string[],key:string,mdate:date,publisher:string[],title:string,url:string,year:int,conf_type:string,author_id:ID
i64,str,str,str,str,str,str,str,str,str,i64,str,i64
27960,null,"""Amir Hossein Alavi""","""https://doi.org/10.1007/978-3-…","""978-3-319-20882-4""","""reference/genetic/2015""","""2020-03-27""","""Springer""","""Handbook of Genetic Programmin…","""db/reference/genetic/genetic20…",2015,"""conference""",11604107
27960,null,"""Amir Hossein Gandomi""","""https://doi.org/10.1007/978-3-…","""978-3-319-20882-4""","""reference/genetic/2015""","""2020-03-27""","""Springer""","""Handbook of Genetic Programmin…","""db/reference/genetic/genetic20…",2015,"""conference""",11604108
27960,null,"""Conor Ryan""","""https://doi.org/10.1007/978-3-…","""978-3-319-20882-4""","""reference/genetic/2015""","""2020-03-27""","""Springer""","""Handbook of Genetic Programmin…","""db/reference/genetic/genetic20…",2015,"""conference""",11578328
32826,null,"""Aaron K. Baughman""","""https://doi.org/10.1007/978-3-…","""978-3-319-14997-4""","""books/sp/BGPP2015""","""2017-05-16""","""Springer""","""Multimedia Data Mining and Ana…","""db/books/collections/BGPP2015.…",2015,"""conference""",11602106
32826,null,"""Jia-Yu Pan""","""https://doi.org/10.1007/978-3-…","""978-3-319-14997-4""","""books/sp/BGPP2015""","""2017-05-16""","""Springer""","""Multimedia Data Mining and Ana…","""db/books/collections/BGPP2015.…",2015,"""conference""",11602116


In [6]:
df_conf_edition.lazy().select(
    pl.col("conf_type:string").value_counts()
).collect()


conf_type:string
struct[2]
"{""workshop"",46106}"
"{""conference"",120436}"


In [7]:
# ===================== Workshops and Conferences =============================
col_names = pl.read_csv(f'{DATA}/dblp_inproceedings_header.csv',separator=';').columns

df_work_conf = pl.read_csv(
        f'{DATA}/dblp_inproceedings.csv',
        ignore_errors=True,
        separator=';',
        new_columns=col_names,
    ).sample(n=NUMBER_OF_CONFERENCES, seed=42)
df_work_conf = (
    df_work_conf
    .pipe(drop_many_nulls, threshold=MISSING_THRESHOLD)
    .rename({'inproceedings:ID': ':ID',})
    .filter(pl.col("author:string[]").is_not_null(),
            pl.col("title:string").is_not_null(),
    )
)  

#keep only those papers that are in the conference dataframe and vice versa
intersection = set(df_conf_edition["key:string"].to_list()).intersection(set(df_work_conf["crossref:string[]"].to_list()))
df_conf_edition = df_conf_edition.filter(pl.col("key:string").is_in(intersection))
df_work_conf = df_work_conf.filter(pl.col("crossref:string[]").is_in(intersection))

df_work_conf.head()

:ID,author:string[],booktitle:string,crossref:string[],ee:string[],key:string,mdate:date,pages:string,title:string,url:string,year:int
i64,str,str,str,str,str,str,str,str,str,i64
8251696,"""Anurag Khandelwal|Lloyd Brown|…","""USENIX Security Symposium""","""conf/uss/2020""","""https://www.usenix.org/confere…","""conf/uss/GrubbsKLBL0R20""","""2021-01-29""","""2451-2468""","""Pancake: Frequency Smoothing f…","""db/conf/uss/uss2020.html#Grubb…",2020
8758260,"""Alan Burns 0001|Guillem Bernat…","""EMSOFT""","""conf/emsoft/2003""","""https://doi.org/10.1007/978-3-…","""conf/emsoft/BurnsBB03""","""2022-03-17""","""1-15""","""A Probabilistic Framework for …","""db/conf/emsoft/emsoft2003.html…",2003
10374573,"""Ciprian-Bogdan Chirila""","""SACI""","""conf/saci/2013""","""https://doi.org/10.1109/SACI.2…","""conf/saci/Chirila13""","""2019-10-19""","""55-60""","""A dialog based game component …","""db/conf/saci/saci2013.html#Chi…",2013
9667521,"""Biplab Kumer Sarker|Kuniaki Ue…","""ISPA""","""conf/ispa/2003""","""https://doi.org/10.1007/3-540-…","""conf/ispa/SarkerMHU03""","""2017-05-21""","""273-284""","""Parallel Algorithms for Mining…","""db/conf/ispa/ispa2003.html#Sar…",2003
8220599,"""Mingyang Zhang 0005|Radhika Ni…","""NSDI""","""conf/nsdi/2019""","""https://www.usenix.org/confere…","""conf/nsdi/ZhangMSG19""","""2021-02-02""","""235-254""","""Understanding Lifecycle Manage…","""db/conf/nsdi/nsdi2019.html#Zha…",2019


In [8]:
print('Percentages of missing by column', df_work_conf.null_count() / df_work_conf.shape[0] * 100)

Percentages of missing by column shape: (1, 11)
┌─────┬─────────┬─────────┬─────────┬────────┬────────┬────────┬────────┬────────┬────────┬────────┐
│ :ID ┆ author: ┆ booktit ┆ crossre ┆ ee:str ┆ key:st ┆ mdate: ┆ pages: ┆ title: ┆ url:st ┆ year:i │
│ --- ┆ string[ ┆ le:stri ┆ f:strin ┆ ing[]  ┆ ring   ┆ date   ┆ string ┆ string ┆ ring   ┆ nt     │
│ f64 ┆ ]       ┆ ng      ┆ g[]     ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    ┆ ---    │
│     ┆ ---     ┆ ---     ┆ ---     ┆ f64    ┆ f64    ┆ f64    ┆ f64    ┆ f64    ┆ f64    ┆ f64    │
│     ┆ f64     ┆ f64     ┆ f64     ┆        ┆        ┆        ┆        ┆        ┆        ┆        │
╞═════╪═════════╪═════════╪═════════╪════════╪════════╪════════╪════════╪════════╪════════╪════════╡
│ 0.0 ┆ 0.0     ┆ 0.0     ┆ 0.0     ┆ 3.7375 ┆ 0.0    ┆ 0.0    ┆ 5.3748 ┆ 0.0    ┆ 0.0    ┆ 0.0    │
│     ┆         ┆         ┆         ┆ 78     ┆        ┆        ┆ 41     ┆        ┆        ┆        │
└─────┴─────────┴─────────┴─────────┴──────

In [9]:
# ===================== Journals =============================
df_journ = pl.read_csv(
        f'{DATA}/dblp_journal.csv',
        ignore_errors=True,
        separator=';',
    )
df_journ = (
    df_journ
    .unique()
    .rename({"journal:string": "name:string"})
    .pipe(drop_many_nulls, threshold=MISSING_THRESHOLD)

)  
df_journ

:ID,name:string
i64,str
15423822,"""ACM Trans. Access. Comput."""
15423898,"""J. Appl. Math."""
15423856,"""Ubiquity"""
15424027,"""Informing Sci. Int. J. an Emer…"
15425351,"""J. Autom. Lang. Comb."""
…,…
15425273,"""Found. Trends Theor. Comput. S…"
15424943,"""Simul. Model. Pract. Theory"""
15423452,"""Pattern Recognit."""


In [10]:
df_journ_pub = pl.read_csv(
        f'{DATA}/dblp_journal_published_in.csv',
        ignore_errors=True,
        separator=';',
    )
df_journ_pub = (
    df_journ_pub
    .pipe(drop_many_nulls, threshold=MISSING_THRESHOLD)

)  

print(df_journ_pub.shape)
print(df_journ_pub.head())

(3829457, 2)
shape: (5, 2)
┌───────────┬──────────┐
│ :START_ID ┆ :END_ID  │
│ ---       ┆ ---      │
│ i64       ┆ i64      │
╞═══════════╪══════════╡
│ 70072     ┆ 15423428 │
│ 70073     ┆ 15423428 │
│ 70074     ┆ 15423428 │
│ 70075     ┆ 15423428 │
│ 70076     ┆ 15423428 │
└───────────┴──────────┘


In [11]:
# ===================== PAPERS IN JOURNALS =============================
col_names = pl.read_csv(f'{DATA}/dblp_article_header.csv',separator=';').columns
df_jour_pap = (pl.read_csv(
        f'{DATA}/dblp_article.csv',
        ignore_errors=True,
        separator=';',
        new_columns=col_names,
    )
    .sample(n=NUMBER_OF_JOURNALS, seed=42)
)
df_jour_pap = (df_jour_pap
          .pipe(drop_many_nulls, threshold=MISSING_THRESHOLD)
          .filter(pl.col("author:string[]").is_not_null())
          .filter(pl.col("title:string").is_not_null())
          .rename({"article:ID": ":ID",})
          .filter(pl.col("author:string[]").is_not_null())
        .with_columns(
                    pl.col("url:string[]").str.split("|").list.get(0).alias("url:string")
        )
          )
print("Journals shape",df_jour_pap.shape)
df_jour_pap.head(5)

Journals shape (248068, 13)


:ID,author:string[],ee:string[],journal:string,key:string,mdate:date,number:string,pages:string,title:string,url:string[],volume:string,year:int,url:string
i64,str,str,str,str,str,str,str,str,str,i64,i64,str
682555,"""Enguday Ademe Mekonnen|Ermias …","""https://doi.org/10.1080/089934…","""Comput. Sci. Educ.""","""journals/csedu/KassaM22""","""2023-01-31""","""4""","""502-531""","""Computational thinking in the …","""db/journals/csedu/csedu32.html…",32,2022,"""db/journals/csedu/csedu32.html…"
1237689,"""Hideaki Sugawara|Satoru Miyaza…","""https://doi.org/10.1093/nar/gk…","""Nucleic Acids Res.""","""journals/nar/SugawaraM03a""","""2020-05-17""","""13""","""3836-3839""","""Biological SOAP servers and we…","""db/journals/nar/nar31.html#Sug…",31,2003,"""db/journals/nar/nar31.html#Sug…"
1204447,"""Junxia Xiong|Liang Zhao|Ling W…","""https://doi.org/10.1111/jcal.1…","""J. Comput. Assist. Learn.""","""journals/jcal/DaiWPZX24""","""2025-01-08""","""6""","""2901-2916""","""A model for assessing student …","""db/journals/jcal/jcal40.html#D…",40,2024,"""db/journals/jcal/jcal40.html#D…"
2873202,"""Bahana Wiradanti|Rajesri Govin…","""https://doi.org/10.1504/IJBIS.…","""Int. J. Bus. Inf. Syst.""","""journals/ijbis/GovindarajuW15""","""2020-04-25""","""3""","""279-299""","""Factors influencing the willin…","""db/journals/ijbis/ijbis19.html…",19,2015,"""db/journals/ijbis/ijbis19.html…"
2140197,"""Emmanuel Vincent 0001|Nicolas …","""https://arxiv.org/abs/2002.016…","""CoRR""","""journals/corr/abs-2002-01687""","""2020-09-18""",null,null,"""Limitations of weak labels for…","""db/journals/corr/corr2002.html…",null,2020,"""db/journals/corr/corr2002.html…"


In [12]:
print('Percentages of missing by column', df_jour_pap.null_count() / df_jour_pap.shape[0] * 100)

Percentages of missing by column shape: (1, 13)
┌─────┬─────┬─────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ :ID ┆ aut ┆ ee: ┆ journa ┆ key:s ┆ mdate ┆ numbe ┆ pages ┆ title ┆ url:s ┆ volum ┆ year: ┆ url:s │
│ --- ┆ hor ┆ str ┆ l:stri ┆ tring ┆ :date ┆ r:str ┆ :stri ┆ :stri ┆ tring ┆ e:str ┆ int   ┆ tring │
│ f64 ┆ :st ┆ ing ┆ ng     ┆ ---   ┆ ---   ┆ ing   ┆ ng    ┆ ng    ┆ []    ┆ ing   ┆ ---   ┆ ---   │
│     ┆ rin ┆ []  ┆ ---    ┆ f64   ┆ f64   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ f64   ┆ f64   │
│     ┆ g[] ┆ --- ┆ f64    ┆       ┆       ┆ f64   ┆ f64   ┆ f64   ┆ f64   ┆ f64   ┆       ┆       │
│     ┆ --- ┆ f64 ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       │
│     ┆ f64 ┆     ┆        ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       ┆       │
╞═════╪═════╪═════╪════════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ 0.0 ┆ 0.0 ┆ 0.5 ┆ 0.0    ┆ 0.0   ┆ 0.0   

## Final data

### AUTHOR (node)

In [13]:
df_node_author = (
    pl.concat([
        df_work_conf.select(['author:string[]']),
        df_jour_pap.select(['author:string[]']),
    ])
    .with_columns(
        pl.col("author:string[]").str.split("|"),
    )
    .explode("author:string[]") #to have one author per row
    .rename({"author:string[]":"name:string"})
    .unique()
    .join(
        df_aut,
        left_on="name:string",
        right_on="name:string",
        how="left",
    )
    .select([":ID", "name:string"])
)
df_node_author = (
    df_node_author
    .with_columns([
        pl.Series([random.choice(countries) for _ in range(df_node_author.height)]).alias("country_of_birth:string"),
        pl.Series([random_date().isoformat() for _ in range(df_node_author.height)]).alias("date_of_birth:date"),
        pl.Series([random.choice([True, False]) for _ in range(df_node_author.height)]).alias("is_phd:boolean")
    ])
    .pipe(lambda df_: df_.cast(schema_type(df_.columns))) #cast to correct data types
)

df_node_author

:ID,name:string,country_of_birth:string,date_of_birth:date,is_phd:boolean
i32,str,str,date,bool
12266320,"""Satyananda Kashyap""","""Netherlands""",1966-02-09,false
12176299,"""Mitsuo Komura""","""Italy""",1952-11-27,false
11570870,"""Hui Jin 0001""","""Sweden""",1978-11-22,false
14721056,"""Shrikant S. Katre""","""Australia""",1969-04-11,false
12021235,"""Zonghao Li""","""Spain""",1968-11-04,false
…,…,…,…,…
13975228,"""Saradwata Sarkar""","""Germany""",1968-08-08,true
15215708,"""Alice Mchardy""","""Sweden""",1963-10-21,true
11876195,"""Mohammad S. Hashmi""","""United Kingdom""",1988-07-05,true


In [14]:
print('Percentages of missing by column', df_node_author.null_count() / df_node_author.shape[0] * 100)

Percentages of missing by column shape: (1, 5)
┌─────┬─────────────┬─────────────────────────┬────────────────────┬────────────────┐
│ :ID ┆ name:string ┆ country_of_birth:string ┆ date_of_birth:date ┆ is_phd:boolean │
│ --- ┆ ---         ┆ ---                     ┆ ---                ┆ ---            │
│ f64 ┆ f64         ┆ f64                     ┆ f64                ┆ f64            │
╞═════╪═════════════╪═════════════════════════╪════════════════════╪════════════════╡
│ 0.0 ┆ 0.0         ┆ 0.0                     ┆ 0.0                ┆ 0.0            │
└─────┴─────────────┴─────────────────────────┴────────────────────┴────────────────┘


### PAPER (node)

In [15]:
papers_cols = [":ID", "title:string", "year:int", "pages:string", "url:string", 
               "ee:string[]","key:string"]
df_node_paper = (
    pl.concat([
        df_work_conf.select(papers_cols),
        df_jour_pap.select(papers_cols),
    ])
    .unique(subset=[":ID"])
    .with_columns(
                pl.col("ee:string[]").str.split("|").list.get(0).alias("DOI:string"),
                pl.lit("Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod",
                ).alias("abstract:string"),
    )
    .drop("ee:string[]")
    .fill_null("unknown")
    .pipe(lambda df_: df_.cast(schema_type(df_.columns)))
    
)
df_node_paper

:ID,title:string,year:int,pages:string,url:string,key:string,DOI:string,abstract:string
i32,str,i32,str,str,str,str,str
10270522,"""Similarity-Invariant Sketch-Ba…",2014,"""398-414""","""db/conf/eccv/eccv2014-6.html#P…","""conf/eccv/ParuiM14""","""https://doi.org/10.1007/978-3-…","""Lorem ipsum dolor sit amet, co…"
1315532,"""Multiresolution Decomposition …",2011,"""unknown""","""db/journals/ejasp/ejasp2011.ht…","""journals/ejasp/NercessianPA11""","""https://doi.org/10.1155/2011/5…","""Lorem ipsum dolor sit amet, co…"
11265695,"""The PERICLES Process Compiler:…",2016,"""76-83""","""db/conf/webist/webist2016-1.ht…","""conf/webist/Campos-LopezW16""","""https://doi.org/10.5220/000575…","""Lorem ipsum dolor sit amet, co…"
2891252,"""Progression of students' SRL p…",2023,"""100881""","""db/journals/iahe/iahe56.html#H…","""journals/iahe/HatalaNK23""","""https://doi.org/10.1016/j.ihed…","""Lorem ipsum dolor sit amet, co…"
745867,"""Structured Neural Decoding Wit…",2022,"""600-614""","""db/journals/tnn/tnn33.html#DuD…","""journals/tnn/DuDHWH22""","""https://doi.org/10.1109/TNNLS.…","""Lorem ipsum dolor sit amet, co…"
…,…,…,…,…,…,…,…
2357460,"""Entropy-based Discovery of Sum…",2021,"""unknown""","""db/journals/corr/corr2105.html…","""journals/corr/abs-2105-10381""","""https://arxiv.org/abs/2105.103…","""Lorem ipsum dolor sit amet, co…"
8623746,"""Sampling Optimization Trade-Of…",2008,"""444-458""","""db/conf/iccsa/iccsa2008-1.html…","""conf/iccsa/MellesHTS08""","""https://doi.org/10.1007/978-3-…","""Lorem ipsum dolor sit amet, co…"
1694704,"""Unintended effects of open dat…",2023,"""107537""","""db/journals/chb/chb139.html#Li…","""journals/chb/LiuW23""","""https://doi.org/10.1016/j.chb.…","""Lorem ipsum dolor sit amet, co…"


In [16]:
print('Percentages of missing by column', df_node_paper.null_count() / df_node_paper.shape[0] * 100)

Percentages of missing by column shape: (1, 8)
┌─────┬──────────────┬──────────┬─────────────┬────────────┬────────────┬────────────┬─────────────┐
│ :ID ┆ title:string ┆ year:int ┆ pages:strin ┆ url:string ┆ key:string ┆ DOI:string ┆ abstract:st │
│ --- ┆ ---          ┆ ---      ┆ g           ┆ ---        ┆ ---        ┆ ---        ┆ ring        │
│ f64 ┆ f64          ┆ f64      ┆ ---         ┆ f64        ┆ f64        ┆ f64        ┆ ---         │
│     ┆              ┆          ┆ f64         ┆            ┆            ┆            ┆ f64         │
╞═════╪══════════════╪══════════╪═════════════╪════════════╪════════════╪════════════╪═════════════╡
│ 0.0 ┆ 0.0          ┆ 0.0      ┆ 0.0         ┆ 0.0        ┆ 0.0        ┆ 0.0        ┆ 0.0         │
└─────┴──────────────┴──────────┴─────────────┴────────────┴────────────┴────────────┴─────────────┘


### AUTHORED (edge)

In [17]:
df_edge_authored = (
    pl.concat([
        df_work_conf.select([":ID", "author:string[]"]),
        df_jour_pap.select([":ID", "author:string[]"]),
    ])
    .with_columns(
        pl.col("author:string[]").str.split("|"),
    )
    .explode("author:string[]") #to have one author per row
    .rename({"author:string[]":"author:string"})
    .unique()
    .pipe(lambda df_: df_.cast(schema_type(df_.columns)))
    .join(
        df_node_author.select([":ID", "name:string"]).rename({":ID":"author_id:ID"}),
        left_on="author:string",
        right_on="name:string",
        how="left",
    )
    .rename({":ID":":END_ID",
             "author_id:ID":":START_ID",
             })
    .select([":START_ID", ":END_ID"])
    .sort(by=[":END_ID", ":START_ID"])
)
df_edge_authored

:START_ID,:END_ID
i32,i32
11592653,37484
11592990,37640
11592594,37780
11593345,37796
11600374,58415
…,…
12574194,11535330
12696925,11548170
11661117,11548240


In [18]:
print('Percentages of missing by column', df_edge_authored.null_count() / df_edge_authored.shape[0] * 100)

Percentages of missing by column shape: (1, 2)
┌───────────┬─────────┐
│ :START_ID ┆ :END_ID │
│ ---       ┆ ---     │
│ f64       ┆ f64     │
╞═══════════╪═════════╡
│ 0.0       ┆ 0.0     │
└───────────┴─────────┘


### CORRESPONDS_TO (edge)

In [19]:
df_edge_corresponds_to = (
    df_edge_authored
    .unique(subset=[":END_ID"]) #Keep only the first author id per paper
    .rename({":END_ID":":START_ID", ":START_ID":":END_ID"}) #change the direction of the edge
)
df_edge_corresponds_to

:END_ID,:START_ID
i32,i32
11592653,37484
11592990,37640
11592594,37780
11593345,37796
11600374,58415
…,…
11756167,11533321
11940948,11534927
11586693,11535330


In [20]:
print('Percentages of missing by column', df_edge_corresponds_to.null_count() / df_edge_corresponds_to.shape[0] * 100)

Percentages of missing by column shape: (1, 2)
┌─────────┬───────────┐
│ :END_ID ┆ :START_ID │
│ ---     ┆ ---       │
│ f64     ┆ f64       │
╞═════════╪═══════════╡
│ 0.0     ┆ 0.0       │
└─────────┴───────────┘


### CITES (edge)

In [21]:
df_edge_cites = generate_citations(df_node_paper)
df_edge_cites

Citing ids: 4980


Generating Citations: 100%|██████████| 4980/4980 [00:41<00:00, 119.07it/s]


:START_ID,:END_ID
i64,i64
9943521,10106660
9943521,8985070
9446374,3766856
9446374,1839634
9263536,8899929
…,…
2162653,2977706
10381820,2005333
9508957,11300509


### KEYWORD (node)

In [22]:
# Predefined keywords
keywords = [
    "Graph Processing", "Data Quality", "Property Graph", "Machine Learning", "Big Data",
    "Data Mining", "Distributed Systems", "Knowledge Graph", "Semantic Web", "Data Integration",
    "Stream Processing", "Graph Databases", "Query Optimization", "Parallel Computing",
    "Data Visualization", "Graph Algorithms", "Data Cleaning", "Network Analysis",
    "Ontologies", "Metadata Management"
]

# Assign a unique ID to each keyword
keyword_to_id = {kw: f"kw_{i+1}" for i, kw in enumerate(keywords)}

df_node_keyword = pl.DataFrame({
            ":ID": list(keyword_to_id.values()),
            "name:string": list(keyword_to_id.keys()),
        })
df_node_keyword

:ID,name:string
str,str
"""kw_1""","""Graph Processing"""
"""kw_2""","""Data Quality"""
"""kw_3""","""Property Graph"""
"""kw_4""","""Machine Learning"""
"""kw_5""","""Big Data"""
…,…
"""kw_16""","""Graph Algorithms"""
"""kw_17""","""Data Cleaning"""
"""kw_18""","""Network Analysis"""


In [23]:
print('Percentages of missing by column', df_node_keyword.null_count() / df_node_keyword.shape[0] * 100)

Percentages of missing by column shape: (1, 2)
┌─────┬─────────────┐
│ :ID ┆ name:string │
│ --- ┆ ---         │
│ f64 ┆ f64         │
╞═════╪═════════════╡
│ 0.0 ┆ 0.0         │
└─────┴─────────────┘


### HAS_KEYWORD (edge)

In [24]:
df_edge_has_keyword = (
    df_node_paper
    .with_columns(
        pl.Series(
            "keywords", 
            [random.sample(keywords, random.randint(1, 3)) for _ in range(df_node_paper.height)]
        )
    )
    .explode("keywords") #to have one keyword per row
    .with_columns(
        pl.col("keywords").alias("keyword:string"),
        pl.col(":ID").alias("paper_id:ID"),
        pl.col("keywords").replace_strict(keyword_to_id).alias("keyword_id:ID")
    )
    .select(
        "paper_id:ID",
        "keyword_id:ID"
    )
    .rename({"paper_id:ID":":START_ID", 
             "keyword_id:ID":":END_ID"})

)
df_edge_has_keyword

:START_ID,:END_ID
i32,str
10270522,"""kw_19"""
10270522,"""kw_14"""
10270522,"""kw_1"""
1315532,"""kw_18"""
1315532,"""kw_7"""
…,…
1694704,"""kw_17"""
11043089,"""kw_12"""
11043089,"""kw_18"""


In [25]:
print('Percentages of missing by column', df_edge_has_keyword.null_count() / df_edge_has_keyword.shape[0] * 100)

Percentages of missing by column shape: (1, 2)
┌───────────┬─────────┐
│ :START_ID ┆ :END_ID │
│ ---       ┆ ---     │
│ f64       ┆ f64     │
╞═══════════╪═════════╡
│ 0.0       ┆ 0.0     │
└───────────┴─────────┘


### JOURNAL (node)

In [26]:
df_node_journal = (
    df_journ
    .join(
        df_jour_pap.select(["journal:string"]),
        left_on="name:string",
        right_on="journal:string",
        how="inner"    
    )
    .unique(subset=[":ID"])
    .pipe(lambda df_: df_.cast(schema_type(df_.columns)))
)
df_node_journal

:ID,name:string
i32,str
15425375,"""Sci. Ann. Cuza Univ."""
15423541,"""J. Vis."""
15425095,"""Oper. Res. Forum"""
15425357,"""Digit. Libr. Perspect."""
15423791,"""Artif. Life"""
…,…
15423627,"""Int. J. Heal. Inf. Syst. Infor…"
15425330,"""J. Comput. Civ. Eng."""
15424684,"""Appl. Math. Comput."""


In [27]:
print('Percentages of missing by column', df_node_journal.null_count() / df_node_journal.shape[0] * 100)

Percentages of missing by column shape: (1, 2)
┌─────┬─────────────┐
│ :ID ┆ name:string │
│ --- ┆ ---         │
│ f64 ┆ f64         │
╞═════╪═════════════╡
│ 0.0 ┆ 0.0         │
└─────┴─────────────┘


### HAS_VOLUME (edge)

In [28]:
df_jorunal_volume = (
    df_jour_pap
    .select(["journal:string","volume:string", "year:int","mdate:date"])
    .unique()
    .rename({"journal:string": "name:string",
             "volume:string":"volume:int",
             "year:int":"year_paper:int"
             })
    .pipe(lambda df_: df_.cast(schema_type(df_.columns)))
    .with_columns(
        pl.col("volume:int").fill_null(1),
        pl.when(pl.col("mdate:date").is_not_null())
        .then(pl.col("mdate:date").dt.year().cast(pl.Int32))
        .otherwise(pl.col("year_paper:int"))
        .alias("year:int")

    )
    .pipe(lambda df_: df_.cast(schema_type(df_.columns)))
    .sort(["mdate:date", "volume:int", "name:string"])
    .drop(["year_paper:int", "mdate:date"])
    .sort([ "year:int", "name:string","volume:int",])
)
df_jorunal_volume = (
        df_jorunal_volume
        .with_columns(
            #add an id jvolume_id:int to each journal volume
            pl.Series([f"jvolume_{i+1}" for i in range(df_jorunal_volume.shape[0])]).alias(":ID"),
        )
        .select(
            ":ID",
            "name:string",
            "volume:int",
            "year:int",
        )
        .join(
            df_node_journal.rename({":ID":"jorunal_id:ID"}),
            on="name:string",
            how="left"
        )
)

df_jorunal_volume

:ID,name:string,volume:int,year:int,jorunal_id:ID
str,str,i32,i32,i32
"""jvolume_1""","""Datenbank Rundbrief""",12,2002,15425024
"""jvolume_2""","""EMISA Forum""",4,2002,15424734
"""jvolume_3""","""EMISA Forum""",5,2002,15424734
"""jvolume_4""","""EMISA Forum""",6,2002,15424734
"""jvolume_5""","""GI Datenbank Rundbrief""",13,2002,15425023
…,…,…,…,…
"""jvolume_119959""","""npj Digit. Medicine""",8,2025,15423740
"""jvolume_119960""","""npj Digit. Medicine""",8,2025,15423740
"""jvolume_119961""","""npj Digit. Medicine""",8,2025,15423740


In [29]:
df_edge_has_volume = (
    df_jorunal_volume
    .select(["jorunal_id:ID",":ID"])
    .rename({":ID":":END_ID", 
             "jorunal_id:ID":":START_ID"})
)
df_edge_has_volume

:START_ID,:END_ID
i32,str
15425024,"""jvolume_1"""
15424734,"""jvolume_2"""
15424734,"""jvolume_3"""
15424734,"""jvolume_4"""
15425023,"""jvolume_5"""
…,…
15423740,"""jvolume_119959"""
15423740,"""jvolume_119960"""
15423740,"""jvolume_119961"""


### JORUNAL VOLUME (node)

In [30]:
df_node_jorunal_volume = df_jorunal_volume.drop(["jorunal_id:ID"])
df_node_jorunal_volume

:ID,name:string,volume:int,year:int
str,str,i32,i32
"""jvolume_1""","""Datenbank Rundbrief""",12,2002
"""jvolume_2""","""EMISA Forum""",4,2002
"""jvolume_3""","""EMISA Forum""",5,2002
"""jvolume_4""","""EMISA Forum""",6,2002
"""jvolume_5""","""GI Datenbank Rundbrief""",13,2002
…,…,…,…
"""jvolume_119959""","""npj Digit. Medicine""",8,2025
"""jvolume_119960""","""npj Digit. Medicine""",8,2025
"""jvolume_119961""","""npj Digit. Medicine""",8,2025


### PUBLISHED_IN jorunalVolume (edge)

In [31]:
df_edge_published_in_j = (
    df_jour_pap
    .select([":ID", "journal:string","volume:string", "year:int","mdate:date"])
    .rename({"journal:string": "name:string",
             "volume:string":"volume:int",
             "year:int":"year_paper:int",
             ":ID":"paper_id:ID"
             })
    .pipe(lambda df_: df_.cast(schema_type(df_.columns)))
    .with_columns(
        pl.col("volume:int").fill_null(1),
        pl.when(pl.col("mdate:date").is_not_null())
        .then(pl.col("mdate:date").dt.year().cast(pl.Int32))
        .otherwise(pl.col("year_paper:int"))
        .alias("year:int")
    )
    .pipe(lambda df_: df_.cast(schema_type(df_.columns)))
    .sort(["mdate:date", "volume:int", "name:string"])
    .drop(["year_paper:int", "mdate:date"])
    .sort([ "year:int", "name:string","volume:int",])
    .join(
        df_jorunal_volume.rename({":ID":"jvolume_id:ID"}),
        on=["name:string", "volume:int", "year:int"],
        how="inner"
    )
    .select(
        "paper_id:ID",
        "jvolume_id:ID"
    )
    .rename({"paper_id:ID":":START_ID", 
             "jvolume_id:ID":":END_ID"})

)
df_edge_published_in_j

:START_ID,:END_ID
i32,str
3284708,"""jvolume_1"""
3284945,"""jvolume_1"""
2859503,"""jvolume_2"""
2859440,"""jvolume_3"""
2859484,"""jvolume_4"""
…,…
463619,"""jvolume_119959"""
463619,"""jvolume_119960"""
463619,"""jvolume_119961"""


### PUBLISHED_IN CONFERENCE (edge)

In [32]:
df_edge_published_in_c = (
    df_conf_edition
    .select([":ID","mdate:date", "key:string", "url:string", "conf_type:string"])
    .unique()
    .rename({":ID":"conf_id:ID"})
    .join(
        df_work_conf.select([":ID","crossref:string[]"]).rename({":ID":"paper_id:ID"}).unique(),
        left_on="key:string",
        right_on="crossref:string[]",
        how="inner"
    )
)

df_edge_published_in_c

conf_id:ID,mdate:date,key:string,url:string,conf_type:string,paper_id:ID
i64,str,str,str,str,i64
7688793,"""2023-06-22""","""conf/interspeech/2000""","""db/conf/interspeech/interspeec…","""conference""",7689054
8986205,"""2019-10-16""","""conf/icdm/2008w""","""db/conf/icdm/icdmw2008.html""","""workshop""",8987893
10338183,"""2023-06-06""","""conf/iceis/2022-2""","""db/conf/iceis/iceis2022-2.html""","""conference""",10334823
10976464,"""2008-12-04""","""conf/acmidc/2008""","""db/conf/acmidc/idc2008.html""","""conference""",10976375
8489890,"""2021-04-09""","""conf/ecai/2020""","""db/conf/ecai/ecai2020.html""","""conference""",8492256
…,…,…,…,…,…
9347288,"""2019-10-16""","""conf/pimrc/2015""","""db/conf/pimrc/pimrc2015.html""","""conference""",9348223
10012131,"""2020-09-22""","""conf/dgo/2020""","""db/conf/dgo/dgo2020.html""","""conference""",10012419
10710415,"""2019-10-16""","""conf/fie/2012""","""db/conf/fie/fie2012.html""","""conference""",10711519


In [33]:
df_conf_ed = df_edge_published_in_c.clone()

In [34]:
df_edge_published_in_c = (
    df_edge_published_in_c
    .select(['paper_id:ID', 'conf_id:ID'])
    .rename({"paper_id:ID":":START_ID", 
             "conf_id:ID":":END_ID"})
)
df_edge_published_in_c

:START_ID,:END_ID
i64,i64
7689054,7688793
8987893,8986205
10334823,10338183
10976375,10976464
8492256,8489890
…,…
9348223,9347288
10012419,10012131
10711519,10710415


### HAS_EDITION WORKSHOP (edge)

In [35]:
df_edge_has_edition_w = (
    df_conf_ed
    .filter(pl.col("conf_type:string").str.to_lowercase().str.contains("workshop"))
    .select(['conf_id:ID'])
    .unique()
)
df_edge_has_edition_w = (
    df_edge_has_edition_w
    .with_columns(
        pl.Series([f"w_{i+1}" for i in range(df_edge_has_edition_w.shape[0])]).alias(":START_ID")
    )
    .rename({"conf_id:ID":":END_ID"})
    .select(":START_ID",":END_ID") #change direction of the edge
)
df_edge_has_edition_w

:START_ID,:END_ID
str,i64
"""w_1""",9559528
"""w_2""",10225693
"""w_3""",9870464
"""w_4""",10786197
"""w_5""",10790907
…,…
"""w_14164""",11255828
"""w_14165""",10783800
"""w_14166""",9765452


### WORKSHOP (node)

In [36]:
df_node_workshop = (
    df_edge_has_edition_w
    .select(":START_ID")
    .rename({":START_ID":":ID"})
)
df_node_workshop

:ID
str
"""w_1"""
"""w_2"""
"""w_3"""
"""w_4"""
"""w_5"""
…
"""w_14164"""
"""w_14165"""
"""w_14166"""


### HAS_EDITION CONFERENCE (edge)

In [37]:
df_edge_has_edition_c = (
    df_conf_ed
    .filter(pl.col("conf_type:string").str.to_lowercase().str.contains("conference"))
    .select(['conf_id:ID'])
    .unique()
)
df_edge_has_edition_c = (
    df_edge_has_edition_c
    .with_columns(
        pl.Series([f"c_{i+1}" for i in range(df_edge_has_edition_c.shape[0])]).alias(":START_ID")
    )
    .rename({"conf_id:ID":":END_ID"})
    .select(":START_ID",":END_ID") #change direction of the edge
)
df_edge_has_edition_c

:START_ID,:END_ID
str,i64
"""c_1""",8746586
"""c_2""",10924888
"""c_3""",9901782
"""c_4""",8853426
"""c_5""",10244039
…,…
"""c_44752""",10985392
"""c_44753""",10284268
"""c_44754""",8921293


### CONFERENCE (node)

In [38]:
df_node_conf = (
    df_edge_has_edition_c
    .select(":START_ID")
    .rename({":START_ID":":ID"})
)
df_node_conf

:ID
str
"""c_1"""
"""c_2"""
"""c_3"""
"""c_4"""
"""c_5"""
…
"""c_44752"""
"""c_44753"""
"""c_44754"""


### HELD_AT (edge)

In [39]:
city_country_pairs = [
    ("New York", "United States"), ("London", "United Kingdom"), ("Barcelona", "Spain"),
    ("Tokyo", "Japan"), ("Berlin", "Germany"), ("Paris", "France"),
    ("Stockholm", "Sweden"), ("Sydney", "Australia"), ("Toronto", "Canada"),
    ("Beijing", "China"), ("Amsterdam", "Netherlands"), ("Zurich", "Switzerland"),
    ("Vienna", "Austria"), ("Milan", "Italy"), ("Seoul", "South Korea"),
    ("Bangalore", "India"), ("São Paulo", "Brazil"), ("Cape Town", "South Africa"),
    ("Tel Aviv", "Israel"), ("Dublin", "Ireland"), ("Boston", "United States"),
    ("San Francisco", "United States"), ("Singapore", "Singapore"), ("Munich", "Germany"),
    ("Stockholm", "Sweden"), ("Dubai", "United Arab Emirates"), ("Moscow", "Russia"),
    ("Hong Kong", "China"), ("Geneva", "Switzerland"), ("Lisbon", "Portugal")
]

df_venue = (
    df_conf_ed
    .select(["conf_id:ID", "mdate:date"])
    .unique()
)
df_venue = (
    df_venue
    .with_columns(
        pl.Series([f"v_{i+1}" for i in range(df_venue.shape[0])]).alias(":ID"),
        pl.Series([random.choice(city_country_pairs) for _ in range(df_venue.shape[0])]).alias("city_country:string[]"),
    )
    .with_columns(
        pl.col("city_country:string[]").list.get(0).alias("city:string"),
        pl.col("city_country:string[]").list.get(1).alias("country:string"),
    )
    .drop("city_country:string[]")
)

df_venue

conf_id:ID,mdate:date,:ID,city:string,country:string
i64,str,str,str,str
8159101,"""2006-11-07""","""v_1""","""Boston""","""United States"""
8376625,"""2021-09-03""","""v_2""","""Stockholm""","""Sweden"""
9256452,"""2024-10-30""","""v_3""","""Tel Aviv""","""Israel"""
10730681,"""2022-12-23""","""v_4""","""Dubai""","""United Arab Emirates"""
9185237,"""2023-06-23""","""v_5""","""San Francisco""","""United States"""
…,…,…,…,…
8594502,"""2019-05-14""","""v_58920""","""São Paulo""","""Brazil"""
7776053,"""2022-04-09""","""v_58921""","""São Paulo""","""Brazil"""
11143091,"""2012-09-18""","""v_58922""","""Toronto""","""Canada"""


In [40]:
df_edge_held_at = (
    df_venue
    .select(["conf_id:ID", ":ID"])
    .rename({":ID":":END_ID", 
             "conf_id:ID":":START_ID"})
)
df_edge_held_at

:START_ID,:END_ID
i64,str
8159101,"""v_1"""
8376625,"""v_2"""
9256452,"""v_3"""
10730681,"""v_4"""
9185237,"""v_5"""
…,…
8594502,"""v_58920"""
7776053,"""v_58921"""
11143091,"""v_58922"""


### VENUE (node)

In [41]:
df_node_venue = (
    df_venue
    .select(":ID", "city:string", "country:string", "mdate:date")
)

df_node_venue

:ID,city:string,country:string,mdate:date
str,str,str,str
"""v_1""","""Boston""","""United States""","""2006-11-07"""
"""v_2""","""Stockholm""","""Sweden""","""2021-09-03"""
"""v_3""","""Tel Aviv""","""Israel""","""2024-10-30"""
"""v_4""","""Dubai""","""United Arab Emirates""","""2022-12-23"""
"""v_5""","""San Francisco""","""United States""","""2023-06-23"""
…,…,…,…
"""v_58920""","""São Paulo""","""Brazil""","""2019-05-14"""
"""v_58921""","""São Paulo""","""Brazil""","""2022-04-09"""
"""v_58922""","""Toronto""","""Canada""","""2012-09-18"""


### CONFERENCE EDITION (node)

In [42]:
df_node_conf_ed = (
    df_conf_ed
    .select("conf_id:ID", "key:string")
    .rename({
        "conf_id:ID":":ID",
        "key:string":"name:string"})
    .sort(["name:string"])
    .unique()
)
df_node_conf_ed

:ID,name:string
i64,str
10569804,"""conf/icsr/2022"""
10048607,"""conf/im/2009"""
11197847,"""conf/latice/2015"""
8993704,"""conf/mobisys/2004"""
10206428,"""conf/trust/2008"""
…,…
10918218,"""conf/mates/2007"""
7932476,"""conf/aplas/2017"""
9474084,"""conf/iciss/2024"""


### REVIEW (node)

In [43]:
# Create a DataFrame with author ranks
df_auth_rank = (
    df_edge_authored
    .group_by(":START_ID")
    .agg(pl.len().alias("papers:int"))
    .sort("papers:int", descending=True)
)

# Assign a rank to authors
df_auth_rank = (
    df_auth_rank
    .with_columns(
        pl.Series([i + 1 for i in range(df_auth_rank.shape[0])]).alias("rank"),
    )
    .rename({":START_ID": "author_id"})
)

# Keep only the top 50 most famous authors as possible reviewers
possible_reviewers = df_auth_rank.select("author_id").to_series().to_list()[:50]

# Group by paper ID and aggregate authors
df_papers_authors = (
    df_edge_authored
    .group_by(":END_ID")  # Group by the paper ID
    .agg(pl.col(":START_ID").alias("authors"))  # Aggregate authors into a list
    .rename({":END_ID": "paper_id"})
)


In [44]:
# Function to generate random reviewers for a given list of authors
def get_random_reviewers(authors, possible_reviewers):
    # Randomly select 1 to 4 reviewers from possible_reviewers excluding the authors
    available_reviewers = [r for r in possible_reviewers if r not in authors]
    num_reviewers = random.randint(1, 5)  # Randomly select between 1 and 4 reviewers
    return random.sample(available_reviewers, num_reviewers)

# Use map_batches to apply the function across the entire dataset
df_papers_authors = (
    df_papers_authors
    .with_columns(
        pl.struct(["authors"]).map_elements(
            lambda row: get_random_reviewers(row["authors"], possible_reviewers),
            return_dtype=pl.List(pl.Int64())  # Return a list of integers (Int64)
        ).alias("reviewers")
    )
    .explode("reviewers")
)

df_papers_authors = df_papers_authors.with_columns(
        pl.Series([f"r_{i+1}" for i in range(df_papers_authors.shape[0])]).alias(":ID"),
    )

df_papers_authors

paper_id,authors,reviewers,:ID
i32,list[i32],i64,str
37484,[11592653],11591618,"""r_1"""
37640,[11592990],11634718,"""r_2"""
37640,[11592990],11710240,"""r_3"""
37780,[11592594],11562437,"""r_4"""
37780,[11592594],11643986,"""r_5"""
…,…,…,…
11548240,"[11661117, 11837960, 13786834]",11573014,"""r_2991340"""
11548240,"[11661117, 11837960, 13786834]",11567381,"""r_2991341"""
11548240,"[11661117, 11837960, 13786834]",11635035,"""r_2991342"""


In [45]:
df_node_review = (
    df_papers_authors.select(":ID")
)
df_node_review

:ID
str
"""r_1"""
"""r_2"""
"""r_3"""
"""r_4"""
"""r_5"""
…
"""r_2991340"""
"""r_2991341"""
"""r_2991342"""


### WRITES (edge)

In [46]:
df_edge_writes = (
    df_papers_authors
    .select('reviewers', ':ID')
    .rename({
        'reviewers': ':START_ID',
        ':ID': ':END_ID'
    })
)
df_edge_writes

:START_ID,:END_ID
i64,str
11591618,"""r_1"""
11634718,"""r_2"""
11710240,"""r_3"""
11562437,"""r_4"""
11643986,"""r_5"""
…,…
11573014,"""r_2991340"""
11567381,"""r_2991341"""
11635035,"""r_2991342"""


### OF (edge)

In [47]:
df_edge_of = (
    df_papers_authors
    .select(':ID','paper_id')
    .rename({
        'paper_id': ':END_ID',
        ':ID': ':START_ID'
    })
)
df_edge_of

:START_ID,:END_ID
str,i32
"""r_1""",37484
"""r_2""",37640
"""r_3""",37640
"""r_4""",37780
"""r_5""",37780
…,…
"""r_2991340""",11548240
"""r_2991341""",11548240
"""r_2991342""",11548240


## Export to csv

In [48]:
# Exporte nodes to CSV
df_node_author.write_csv(f'final_data/node_author.csv', separator=';')
df_node_conf.write_csv(f'final_data/node_conf.csv', separator=';')
df_node_conf_ed.write_csv(f'final_data/node_conf_edition.csv', separator=';')
df_node_journal.write_csv(f'final_data/node_journal.csv', separator=';')
df_node_jorunal_volume.write_csv(f'final_data/node_journal_volume.csv', separator=';')
df_node_keyword.write_csv(f'final_data/node_keyword.csv', separator=';')
df_node_paper.write_csv(f'final_data/node_paper.csv', separator=';')
df_node_venue.write_csv(f'final_data/node_venue.csv', separator=';')
df_node_workshop.write_csv(f'final_data/node_workshop.csv', separator=';')
df_node_review.write_csv(f'final_data/node_review.csv', separator=';')
# Export edges to CSV
df_edge_authored.write_csv(f'final_data/edge_authored.csv', separator=';')  
df_edge_corresponds_to.write_csv(f'final_data/edge_corresponds_to.csv', separator=';')  
df_edge_has_edition_c.write_csv(f'final_data/edge_has_edition_c.csv', separator=';')
df_edge_has_edition_w.write_csv(f'final_data/edge_has_edition_w.csv', separator=';')
df_edge_has_keyword.write_csv(f'final_data/edge_has_keyword.csv', separator=';')
df_edge_has_volume.write_csv(f'final_data/edge_has_volume.csv', separator=';')
df_edge_held_at.write_csv(f'final_data/edge_held_at.csv', separator=';')
df_edge_published_in_c.write_csv(f'final_data/edge_published_in_c.csv', separator=';')
df_edge_published_in_j.write_csv(f'final_data/edge_published_in_j.csv', separator=';')
df_edge_cites.write_csv(f'final_data/edge_cites.csv', separator=';')
df_edge_writes.write_csv(f'final_data/edge_writes.csv', separator=';')
df_edge_of.write_csv(f'final_data/edge_of.csv', separator=';')

## Load CSVs into Neo4j Database

1. Once the csv are created, copy all the files into the import folder of the database. To acces this final destination path, follow these steps:
    1. Open Neo4j Desktop browser
    2. Go to your project.
    3. Click the three dots of the databse, and then click on Terminal.
    4. Inside the terminal, execute the following:
        ```bash
            cd import
            pwd
        ```
        In here, you should see a path like. Copy it
        ```bash
            /Users/<user>/Library/Application\ Support/Neo4j\ Desktop/Application/relate-data/dbmss/dbms-4ab0220b-953c-404a-b291-d5650f203d5b/import
        ```
    5. Open a terminal on your computer and go to the repository `final_data` folder. From inside that folder, execute the following command to copy the generated csv into Neo4j import folder.
        ```bash
            cp * /Users/<user>/Library/Application\ Support/Neo4j\ Desktop/Application/relate-data/dbmss/dbms-4ab0220b-953c-404a-b291-d5650f203d5b/import
        ```
    6. Finally, go again to Neo4J Desktop and open a terminal of your database (as mentioned in step 3). Execute the following import to load the data in the databse:
        ```bash
            bin/neo4j-admin database import full neo4j \
            --delimiter=";" \
            --array-delimiter="|" \
            --nodes=Author=import/node_author.csv \
            --nodes=Journal=import/node_journal.csv \
            --nodes=Paper=import/node_paper.csv \
            --nodes=Conference=import/node_conf.csv \
            --nodes=JournalVolume=import/node_journal_volume.csv \
            --nodes=Venue=import/node_venue.csv \
            --nodes=ConferenceEdition=import/node_conf_edition.csv \
            --nodes=Keyword=import/node_keyword.csv \
            --nodes=Workshop=import/node_workshop.csv \
            --nodes=Review=import/node_review.csv \
            --relationships=AUTHORED=import/edge_authored.csv \
            --relationships=HAS_EDITION_W=import/edge_has_edition_w.csv \
            --relationships=HAS_EDITION_C=import/edge_has_edition_c.csv \
            --relationships=HAS_VOLUME=import/edge_has_volume.csv \
            --relationships=PUBLISHED_IN_J=import/edge_published_in_j.csv \
            --relationships=PUBLISHED_IN_C=import/edge_published_in_c.csv \
            --relationships=CITES=import/edge_cites.csv \
            --relationships=HAS_KEYWORD=import/edge_has_keyword.csv \
            --relationships=CORRESPONDS_TO=import/edge_corresponds_to.csv \
            --relationships=HELD_AT=import/edge_held_at.csv \
            --relationships=WRITES=import/edge_writes.csv \
            --relationships=OF=import/edge_of.csv \
            --verbose \
            --skip-duplicate-nodes \
            --overwrite-destination
        ```
    
    After these steps, your databse should be populated with data. Initialize it, open the browser and execute the following cypher command to see the database schema

    ![image.png](imgs/check_data_population.png)

# A.3.
---

To extend the model and show the felxibility of a graph database, we added two more properties to the `REVIEW` node. 
Also, we created new fake `INSTITUTION` node, with id, type and name properties. Finally, we randomly assigned the existing authors to an institution, keeping only the ids for the edge file.

![BDM_lab1_a3.png](imgs/BDM_lab01_a3.png)

### REVIEW (node) Extended

In [49]:
df_node_review = (
    df_node_review
    .with_columns(
        pl.lit("Lorem ipsum dolor sit amet, consectetur adipiscing elit, sed do eiusmod").alias("content:string"),
        pl.lit(True).alias("approved:boolean"),
    )
)
df_node_review

:ID,content:string,approved:boolean
str,str,bool
"""r_1""","""Lorem ipsum dolor sit amet, co…",true
"""r_2""","""Lorem ipsum dolor sit amet, co…",true
"""r_3""","""Lorem ipsum dolor sit amet, co…",true
"""r_4""","""Lorem ipsum dolor sit amet, co…",true
"""r_5""","""Lorem ipsum dolor sit amet, co…",true
…,…,…
"""r_2991340""","""Lorem ipsum dolor sit amet, co…",true
"""r_2991341""","""Lorem ipsum dolor sit amet, co…",true
"""r_2991342""","""Lorem ipsum dolor sit amet, co…",true


### INSTITUTION (node)

In [54]:
intutitions_type_pairs = [
    ("UCLA", "university"), ("BSE", "university"), ("UPF", "university"),
    ("UBA", "university"), ("Alphabet Inc.", "company"), ("Meta", "company"),
    ("OpenAI", "company"), ("Microsoft", "company")
]

#creat a dataframe with intutitions_type_pairs as columns name and type
df_node_institution = (
    pl.DataFrame({
        "name:string": [pair[0] for pair in intutitions_type_pairs],
        "type:string": [pair[1] for pair in intutitions_type_pairs],
    })
    .with_columns(
        pl.Series([f"inst_{i+1}" for i in range(len(intutitions_type_pairs))]).alias(":ID"),
    )
    .select(":ID", "name:string", "type:string")
)
df_node_institution

:ID,name:string,type:string
str,str,str
"""inst_1""","""UCLA""","""university"""
"""inst_2""","""BSE""","""university"""
"""inst_3""","""UPF""","""university"""
"""inst_4""","""UBA""","""university"""
"""inst_5""","""Alphabet Inc.""","""company"""
"""inst_6""","""Meta""","""company"""
"""inst_7""","""OpenAI""","""company"""
"""inst_8""","""Microsoft""","""company"""


### AFFILIATED_TO (edge)

In [56]:
institutions_ids = df_node_institution[":ID"].to_list()
df_edge_affiliated_to = (
    df_node_author
    .select(":ID", "name:string")
    .with_columns(
        pl.Series([random.choice(institutions_ids) for _ in range(df_node_author.shape[0])]).alias("inst_id:ID"),
    )
    .rename({
        ":ID": ":START_ID",
        "inst_id:ID": ":END_ID"
    })
    .select(":START_ID", ":END_ID")
)
df_edge_affiliated_to

:START_ID,:END_ID
i32,str
12266320,"""inst_3"""
12176299,"""inst_7"""
11570870,"""inst_8"""
14721056,"""inst_6"""
12021235,"""inst_4"""
…,…
13975228,"""inst_7"""
15215708,"""inst_8"""
11876195,"""inst_4"""


## Export new data and load

In [59]:
df_node_review.write_csv(f'final_data/node_review_modified.csv', separator=';')
df_edge_affiliated_to.write_csv(f'final_data/edge_affiliated_to.csv', separator=';')
df_node_institution.write_csv(f'final_data/node_institution.csv', separator=';')

To load the new data, follow the `Load CSVs into Neo4j Database` instructions but in step 6 run the following:
```bash
  bin/neo4j-admin database import incremental neo4j \
    --delimiter=";" \
    --array-delimiter="|" \
    --nodes=Review=import/node_review_modified.csv \
    --nodes=Institution=import/node_institution.csv \
    --relationships=AFFILIATED_TO=import/edge_affiliated_to.csv \
    --verbose \
    --skip-duplicate-nodes \
    --force
```


# B.
---

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query: str):
    with driver.session() as session:
        session.execute_write(lambda tx: tx.run(query))

def test_connection():
    try:
        with driver.session() as session:
            result = session.run("RETURN 1")
            value = result.single()[0]
            if value == 1:
                print("✅ Connected to Neo4j!")
            else:
                print("❌ Unexpected response.")
    except Exception as e:
        print(f"❌ Connection failed: {e}")

# Run the test
test_connection()

## B.1. Top 3 most‑cited papers per conference

## B.2. 

In [ ]:
cypher_query = """
    MATCH 
    (conf:Conference)-[:HAS_EDITION]->(ed:ConferenceEdition),
    (ed)<-[:PUBLISHED_IN]-(p:Paper)<-[:AUTHORED]-(a:Author)
    WITH 
    conf.name          AS conference,
    a.full_name        AS author,
    COUNT(DISTINCT ed.edition_id) AS editionsCount
    WHERE 
    editionsCount >= 4
    RETURN 
    conference,
    author,
    editionsCount
    ORDER BY 
    conference,
    editionsCount DESC;
"""
result = run_query(cypher_query)

## B.3.

In [ ]:
cypher_query = """
    // 1) Gather, per journal, all papers published in 2022 or 2023
    MATCH (j:Journal)-[:HAS_VOLUME]->(vol:JournalVolume)<-[:PUBLISHED_IN]-(p:Paper)
    WHERE vol.year IN [2022, 2023]
    WITH j, collect(p) AS prevPapers, size(collect(p)) AS numPrevPapers

    // 2) Count incoming citations in 2024 to any of those papers
    UNWIND prevPapers AS oldPaper
    OPTIONAL MATCH (citing:Paper)-[:CITES]->(oldPaper)
    WHERE citing.year = 2024
    WITH j, numPrevPapers, count(citing) AS numCitations

    // 3) Compute impact factor, guard against division by zero
    RETURN
    j.name            AS Journal,
    numCitations      AS Citations2024,
    numPrevPapers     AS Papers2022_23,
    CASE
        WHEN numPrevPapers > 0
        THEN toFloat(numCitations) / numPrevPapers
        ELSE NULL
    END               AS ImpactFactor2024
    ORDER BY ImpactFactor2024 DESC;
"""
result = run_query(cypher_query)

## B.4.

In [ ]:
cypher_query = """
    // 1) Compute per‑paper citation counts
    MATCH (a:Author)-[:AUTHORED]->(p:Paper)
    OPTIONAL MATCH (p)<-[:CITES]-()
    WITH a, p, count(*) AS citations

    // 2) Gather all that author's citation counts into a list
    WITH a, collect(citations) AS citationList

    // 3) Sort ascending via APOC, then reverse via list comprehension
    WITH 
    a.full_name AS Author, 
    apoc.coll.sort(citationList)      AS ascList
    WITH 
    Author, 
    [ i IN range(size(ascList)-1, 0, -1) | ascList[i] ] AS sortedList

    // 4) Unwind positions and keep only those where citationCount >= rank
    UNWIND range(0, size(sortedList)-1) AS idx
    WITH 
    Author, 
    idx + 1             AS h, 
    sortedList[idx]     AS citationCount
    WHERE citationCount >= h

    // 5) The h‑index is the maximum valid h, defaulting to 0 if none
    RETURN 
    Author, 
    coalesce(max(h), 0) AS HIndex
    ORDER BY HIndex DESC, Author;
"""
result = run_query(cypher_query)

# C.
---

Run algorithms for part C and check results.